# 13 — OpenTofu Infrastructure: Dual-Mode Architecture

This notebook describes the Nix/Terranix-driven OpenTofu infrastructure strategy
for this ML deployment reference platform.

All infrastructure is **generated from Nix expressions** (via Terranix) and compiled
to OpenTofu-compatible `*.tf.json` files. Two deployment profiles are supported:

| Profile | Purpose | Key services |
|---|---|---|
| `local_emulation` | Verify entire stack locally before cloud | LocalStack, K3s, Slurm-Docker, MinIO |
| `cloud` | Production-grade cloud deployment | AWS (S3, EKS, RDS), Lambda.ai Slurm |

**Rule**: Never apply the `cloud` profile until `local_emulation` has passed end-to-end verification.


## Architecture overview

```{mermaid}
graph TD
  F[flake.nix] --> T[Terranix]
  D[devenv.nix] --> T
  T --> LJ[infra/local/main.tf.json]
  T --> CJ[infra/cloud/main.tf.json]
  LJ --> LT[tofu apply local]
  CJ --> CT[tofu apply cloud]
  LT --> LS[LocalStack]
  LT --> K3[K3s Docker]
  LT --> SD[Slurm Docker]
  CT --> AW[AWS S3/EKS/RDS]
  CT --> LA[Lambda.ai Slurm]
```


## Local emulation stack

The local emulation stack mirrors every service boundary present in the cloud topology.
It is split across two Docker Compose files:

### Data plane (`docker-compose.dev.yml`)

| Service | Port | Role |
|---|---|---|
| PostgreSQL | 5432 | MLflow backend store |
| MinIO | 9000/9001 | S3-compatible artifact store |
| MLflow | 5001 | Experiment tracking UI + API |

### Compute plane (`docker-compose.local-infra.yml`)

| Service | Port | Emulates |
|---|---|---|
| LocalStack | 4566 | AWS APIs (S3, IAM, EC2, STS, Secrets Manager) |
| K3s | 6443 | Kubernetes API (EKS) |
| slurmctld | 6817 | Slurm controller (Lambda.ai head node) |
| slurmd | — | Slurm worker (Lambda.ai compute nodes) |

A `localstack_init` container creates S3 buckets and an IAM role on startup.


## Nix/Terranix module structure

```
nix/
  modules/
    shared.nix   # mlDeploy options + S3 bucket resources (both profiles)
    local.nix    # local_emulation overrides
    cloud.nix    # cloud overrides (sensitive vars injected at apply time)
  profiles/
    local.nix    # Terranix entrypoint: imports modules/local.nix
    cloud.nix    # Terranix entrypoint: imports modules/cloud.nix
```

### Generating OpenTofu JSON from flake.nix

```bash
# Generate local profile
nix build .#localInfraJson && cp result infra/local/main.tf.json

# Generate cloud profile
nix build .#cloudInfraJson && cp result infra/cloud/main.tf.json
```

The `flake.nix` outputs are defined via `terranix.lib.terranixConfiguration`.


## Profile switching and verification workflow

```{mermaid}
flowchart LR
  A[Start local stack] --> B[tofu apply local]
  B --> C{Tests pass?}
  C -- No --> D[Fix and retry]
  D --> B
  C -- Yes --> E[Mark local verified]
  E --> F[tofu apply cloud]
```

### Step-by-step

1. `docker compose -f docker-compose.dev.yml up -d`
2. `docker compose -f docker-compose.local-infra.yml up -d`
3. `nix build .#localInfraJson`
4. `cd infra/local && tofu init && tofu apply`
5. `uv run python -m unittest discover -s tests -q`
6. `curl http://localhost:5001/health`
7. `aws --endpoint-url=http://localhost:4566 s3 ls`
8. `kubectl --kubeconfig /output/kubeconfig.yaml get nodes`
9. `scontrol ping`
10. Only when all checks pass: apply cloud profile.


## Trade-offs

| Aspect | Local emulation | Cloud |
|---|---|---|
| Fidelity | High for API shape; lower for scale | Full production fidelity |
| Cost | Zero (Docker only) | AWS/Lambda.ai billing |
| Speed | Fast iteration (seconds) | Slow (minutes per apply) |
| Secrets | Dummy credentials | Real secrets via env/Secrets Manager |
| GPU | CPU-only (emulated) | Real GPU nodes on Lambda.ai |
| K8s | K3s single-node | EKS multi-node |
| Slurm | 1 controller + 1 worker Docker | Lambda.ai multi-node cluster |

**LocalStack Community limitations**:

- EKS API is Pro — use K3s directly for local Kubernetes.
- S3, IAM, STS, Secrets Manager, CloudWatch Logs are in Community.
- SageMaker not relevant (we use MLflow + Lambda.ai).


## Security considerations

### Local emulation

- All credentials are dummy values (`access_key = "test"`, `secret_key = "test"`).
- Never use local emulation credentials in cloud contexts.
- K3s runs privileged — keep compute-plane Compose off production hosts.

### Cloud profile

- No secrets in `nix/modules/cloud.nix`; all sensitive values are Terraform variables.
- Inject via `TF_VAR_*` environment variables, AWS Secrets Manager, or CI secret store.
- IAM roles follow least-privilege.
- Terraform state for cloud profile should be stored in S3 with DynamoDB locking
  (add `terraform.backend` block in `nix/modules/cloud.nix`).


## Implementation steps for engineers

1. **Add Terranix to `flake.nix`**:
   ```nix
   inputs.terranix.url = "github:terranix/terranix";
   ```
   Define `packages.localInfraJson` and `packages.cloudInfraJson` outputs.

2. **Install OpenTofu**: add `pkgs.opentofu` to `devShells.default.buildInputs`.

3. **Create `infra/` directory structure**:
   ```
   infra/
     local/   # receives localInfraJson build output
     cloud/   # receives cloudInfraJson build output
   ```

4. **Run local verification** (workflow steps 1-9 above).

5. **CI integration**: GitHub Actions job that starts Compose stacks, builds local JSON,
   runs `tofu plan`, runs test suite, and blocks cloud apply until local job passes.
